### Reducers

Great question! **Reducers** in LangGraph control **how state updates are combined** when multiple nodes update the same key.

---

## The Problem Reducers Solve

By default, state updates **override** previous values:

```python
class State(TypedDict):
    messages: str

def node1(state):
    return {"messages": "Hello"}

def node2(state):
    return {"messages": "World"}

# Result: {"messages": "World"}  ← "Hello" is LOST!
```

**Reducers** let you change this behavior - instead of overriding, you can **append, merge, sum**, etc.

---

## How to Use Reducers

You use the `Annotated` type to specify a reducer function:

```python
from typing import Annotated
from typing_extensions import TypedDict
from operator import add

class State(TypedDict):
    messages: Annotated[list[str], add]  # Use 'add' reducer
```

Now when nodes return updates, they **combine** instead of override!

---

## Common Reducers

### 1. **`operator.add`** - For Lists (Append)

```python
from operator import add
from typing import Annotated

class State(TypedDict):
    messages: Annotated[list[str], add]

def node1(state):
    return {"messages": ["Hello"]}

def node2(state):
    return {"messages": ["World"]}

# Result: {"messages": ["Hello", "World"]}  ← Both preserved!
```

**Use case:** Accumulating messages, logs, history

---

### 2. **Custom Reducer Function** - For Complex Logic

```python
def merge_dicts(existing: dict, new: dict) -> dict:
    """Merge two dictionaries"""
    return {**existing, **new}

class State(TypedDict):
    config: Annotated[dict, merge_dicts]

def node1(state):
    return {"config": {"temperature": 0.7}}

def node2(state):
    return {"config": {"max_tokens": 100}}

# Result: {"config": {"temperature": 0.7, "max_tokens": 100}}
```

**Use case:** Merging configurations, combining metadata

---

### 3. **`operator.add`** - For Numbers (Sum)

```python
from operator import add

class State(TypedDict):
    total_cost: Annotated[int, add]

def node1(state):
    return {"total_cost": 10}

def node2(state):
    return {"total_cost": 25}

# Result: {"total_cost": 35}  ← Summed!
```

**Use case:** Tracking totals, counters, scores

---

### 4. **Custom Reducer for Deduplication**

```python
def unique_items(existing: list, new: list) -> list:
    """Keep only unique items"""
    return list(set(existing + new))

class State(TypedDict):
    tags: Annotated[list[str], unique_items]

def node1(state):
    return {"tags": ["python", "AI"]}

def node2(state):
    return {"tags": ["AI", "coding"]}

# Result: {"tags": ["python", "AI", "coding"]}  ← No duplicates
```

---

## Real-World Example: Chat Application

```python
from typing import Annotated
from typing_extensions import TypedDict
from operator import add

class ChatState(TypedDict):
    messages: Annotated[list[dict], add]  # Accumulate all messages
    user_name: str                         # Regular field (override)
    token_count: Annotated[int, add]      # Sum tokens

def user_input(state):
    return {
        "messages": [{"role": "user", "content": "Hello"}],
        "user_name": "Alice",
        "token_count": 5
    }

def ai_response(state):
    return {
        "messages": [{"role": "assistant", "content": "Hi Alice!"}],
        "token_count": 8
    }

# After both nodes:
# {
#     "messages": [
#         {"role": "user", "content": "Hello"},
#         {"role": "assistant", "content": "Hi Alice!"}
#     ],
#     "user_name": "Alice",  ← Last write wins (no reducer)
#     "token_count": 13       ← 5 + 8 = 13 (add reducer)
# }
```

---

## How Reducers Work Internally

```python
# Without reducer (default override):
new_state = {**old_state, **updates}

# With reducer:
new_state = {
    **old_state,
    "key": reducer_function(old_state["key"], updates["key"])
}
```

---

## Custom Reducer Example: Keep Last N Items

```python
def keep_last_n(n: int):
    """Factory function to create a reducer that keeps last N items"""
    def reducer(existing: list, new: list) -> list:
        combined = existing + new
        return combined[-n:]  # Keep only last N items
    return reducer

class State(TypedDict):
    recent_messages: Annotated[list[str], keep_last_n(5)]  # Keep last 5

def node1(state):
    return {"recent_messages": ["msg1", "msg2", "msg3"]}

def node2(state):
    return {"recent_messages": ["msg4", "msg5", "msg6"]}

# Result: {"recent_messages": ["msg2", "msg3", "msg4", "msg5", "msg6"]}
#                                       ↑ Only last 5 kept
```

---

## Comparison: With vs Without Reducers

### Without Reducer (Default Override)
```python
class State(TypedDict):
    items: list[str]

# Node1: {"items": ["A", "B"]}
# Node2: {"items": ["C", "D"]}
# Result: {"items": ["C", "D"]}  ← Lost A, B!
```

### With Reducer (Accumulate)
```python
from operator import add

class State(TypedDict):
    items: Annotated[list[str], add]

# Node1: {"items": ["A", "B"]}
# Node2: {"items": ["C", "D"]}
# Result: {"items": ["A", "B", "C", "D"]}  ← All preserved!
```

---

## When to Use Reducers?

| Use Case | Reducer |
|----------|---------|
| **Accumulate messages/logs** | `operator.add` for lists |
| **Sum costs/scores** | `operator.add` for numbers |
| **Merge configurations** | Custom merge function |
| **Keep conversation history** | `operator.add` for lists |
| **Deduplicate items** | Custom dedup function |
| **Keep latest value only** | No reducer (default) |
| **Track multiple updates** | `operator.add` for lists |

---

## Key Takeaways

1. **Default behavior** = override (last write wins)
2. **Reducers** = custom logic for combining updates
3. **`operator.add`** = most common (lists and numbers)
4. **Custom reducers** = any logic you need
5. **Use `Annotated[type, reducer]`** syntax

Reducers are essential for building intelligent systems where you need to **accumulate information** rather than just replacing it!

